# Laboratorio 4 - RAG y evaluacion

**Duracion estimada:** 45 minutos

**Objetivo**

- Montar un flujo RAG de extremo a extremo.
- Forzar una politica clara de abstencion.
- Evaluar respuestas con preguntas respondibles y no respondibles.

**Prerequisitos**

- Stack Docker levantado
- Modelos disponibles en Ollama
- Coleccion indexada o capacidad de rehacerla

**Criterios de exito**

- Defines `retrieve`, `build_context` y `ask_rag`.
- Respondes una pregunta presente en los documentos.
- Te abstienes razonablemente en una pregunta fuera de los documentos.


## Antes de empezar

Si no estas seguro de que la coleccion existe, puedes rehacer la indexacion desde el laboratorio 3.
En esta solucion se incluye una ayuda para reconstruirla si hace falta.


In [ ]:
import os
import uuid
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
CHAT_MODEL = os.getenv("CHAT_MODEL", "llama3")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))
TOP_K = int(os.getenv("TOP_K", "3"))

OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"
DOCS_DIR = Path("..").resolve() / "docs"
EVAL_PATH = Path("..").resolve() / "evaluation" / "questions.csv"

chat_model = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
client = QdrantClient(url=QDRANT_URL)

print(
    {
        "COLLECTION_NAME": COLLECTION_NAME,
        "TOP_K": TOP_K,
        "EVAL_PATH": str(EVAL_PATH),
    }
)


## Paso 1 - Definir un prompt con abstencion

Construye un prompt que obligue al modelo a:

- usar solo el contexto recuperado;
- decir claramente que no lo sabe si falta informacion.


In [ ]:
prompt = ChatPromptTemplate.from_template(
    '''
Responde solo con la informacion del contexto.
Si el contexto no es suficiente, di claramente que no lo sabes.

Pregunta: {question}

Contexto:
{context}
    '''.strip()
)
chain = prompt | chat_model | StrOutputParser()


## Paso 2 - Implementar retrieval y ensamblado de contexto

Completa estas tres funciones:

- `retrieve(question, top_k=TOP_K)`
- `build_context(hits)`
- `ask_rag(question, top_k=TOP_K)`


In [ ]:
def ensure_collection_ready():
    try:
        info = client.get_collection(COLLECTION_NAME)
        if info.points_count and info.points_count > 0:
            return info
    except Exception:
        pass

    loader = DirectoryLoader(
        str(DOCS_DIR),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)

    for idx, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = idx

    vectors = embedding_model.embed_documents([chunk.page_content for chunk in chunks])
    vector_size = len(vectors[0])

    client.recreate_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )

    points = []
    for chunk, vector in zip(chunks, vectors):
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={
                    "source": chunk.metadata.get("source", "desconocido"),
                    "chunk_id": chunk.metadata["chunk_id"],
                    "text": chunk.page_content,
                },
            )
        )

    client.upsert(collection_name=COLLECTION_NAME, points=points)
    return client.get_collection(COLLECTION_NAME)


ensure_collection_ready()


def retrieve(question: str, top_k: int = TOP_K):
    query_vector = embedding_model.embed_query(question)
    return client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k,
    )


def build_context(hits):
    blocks = []
    for hit in hits:
        blocks.append(
            f"Fuente: {hit.payload.get('source', 'desconocido')} | "
            f"Chunk: {hit.payload.get('chunk_id', '?')}\n"
            f"{hit.payload.get('text', '')}"
        )
    return "\n\n".join(blocks)


def ask_rag(question: str, top_k: int = TOP_K):
    hits = retrieve(question, top_k=top_k)
    context = build_context(hits)
    answer = chain.invoke({"question": question, "context": context})
    return answer, hits, context


### Checkpoint 1

Antes de seguir, deberias tener claro:

- donde se genera el embedding de consulta;
- donde se construye el contexto textual;
- donde se aplica la instruccion de abstencion.


## Paso 3 - Ejecutar una pregunta respondible y una no respondible

Usa estas dos preguntas:

- `Puedo devolver una mesa ya montada si simplemente no me convence?`
- `FrasoHome ofrece servicio de montaje a domicilio?`


In [ ]:
answerable_question = "Puedo devolver una mesa ya montada si simplemente no me convence?"
unanswerable_question = "FrasoHome ofrece servicio de montaje a domicilio?"

for question in [answerable_question, unanswerable_question]:
    answer, hits, _ = ask_rag(question)
    print("=" * 100)
    print("PREGUNTA")
    print(question)
    print("\nRESPUESTA")
    print(answer)
    print("\nFUENTES")
    for hit in hits:
        print(
            {
                "score": round(hit.score, 4),
                "source": hit.payload.get("source"),
                "chunk_id": hit.payload.get("chunk_id"),
            }
        )


## Paso 4 - Evaluar un lote de preguntas

Carga `evaluation/questions.csv`, ejecuta al menos las 5 primeras preguntas y crea un DataFrame con:

- `question`
- `expected_source`
- `top_source`
- `should_abstain`
- `answer`


In [ ]:
df = pd.read_csv(EVAL_PATH).head(5).copy()
results = []

for _, row in df.iterrows():
    answer, hits, _ = ask_rag(row["question"])
    top_source = None
    if hits:
        top_source = Path(hits[0].payload.get("source", "")).name

    results.append(
        {
            "question": row["question"],
            "expected_source": row["expected_source"],
            "top_source": top_source,
            "should_abstain": row["should_abstain"],
            "answer": answer,
        }
    )

pd.DataFrame(results)


## Paso 5 - Comparar una configuracion

Repite una pregunta con `top_k=TOP_K` y luego con `top_k=5`. Observa si cambia:

- la calidad del contexto,
- la respuesta final,
- el riesgo de meter ruido.


In [ ]:
comparison_question = "Que herramientas necesito para montar la mesa Oslo?"

answer_default, hits_default, _ = ask_rag(comparison_question, top_k=TOP_K)
answer_k5, hits_k5, _ = ask_rag(comparison_question, top_k=5)

print("TOP_K por defecto")
print(answer_default)
print([Path(hit.payload.get("source", "")).name for hit in hits_default])

print("\nTOP_K = 5")
print(answer_k5)
print([Path(hit.payload.get("source", "")).name for hit in hits_k5])


### Checkpoint 2

Verifica:

- tienes al menos una respuesta sustentada por el contexto;
- tienes al menos un caso donde el sistema debe abstenerse;
- ya puedes razonar si `TOP_K` mejora recall o introduce ruido.


## Reflexion 1

**Respuesta orientativa**

- Recuperar bien significa traer contexto relevante y suficiente.
- Responder bien significa redactar una salida fiel al contexto y a la instruccion.
- Si falla retrieval, el modelo generativo puede sonar convincente pero apoyarse en contexto pobre.


## Reflexion 2

**Respuesta orientativa**

- El sistema debe abstenerse cuando el contexto no contiene la evidencia necesaria.
- En este repo, la primera mejora a revisar suele ser retrieval y chunking antes que el prompt.

**Mini extension opcional**

Repite la evaluacion despues de reindexar con otra configuracion de chunking y compara resultados.
